# 05 — Top non-government, non-party advocacy spenders

**The question**: among advertisers that are *neither government nor a registered party/candidate*, who spent the most on political-adjacent FB ads around the May 2022 federal election, and how do their spending patterns differ across the campaign window?

**Why exclude parties and candidates**: party central offices and candidate ads dominate raw spend totals (Australian Labor Party + UAP alone account for ~$5M pre-election spend). Their election-only behaviour is expected and uninteresting. The structurally interesting question is who's *in the advocacy ecosystem alongside them* — NGO campaigns, PAC-style election funders, single-issue advocacy groups — and how that ecosystem's spending pattern compares.

**Headline deliverable**: ranked list of the top 20 non-government, non-party advocacy spenders, with pre/post-election spend, persistence ratios, and topic mix.

**Input corpus**: v3 parquet from [04_topic_join.ipynb](04_topic_join.ipynb), filtered down to `match_type IS NULL` (i.e. only the LDA-classified advocacy residual — no candidate, no party_org, no government).

**Window**: 6 months before to 6 months after the 21 May 2022 election.

## 1. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count as spark_count, countDistinct,
    when, lit, date_trunc, desc, mode, broadcast,
)
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

spark = SparkSession.builder \
    .appName('FB_API_election_spenders') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

In [ ]:
V3_PATH = '/user/s3348393/main/preprocessing/v3/parquet'

ELECTION_DATE = pd.Timestamp('2022-05-21')
WINDOW_START  = pd.Timestamp('2021-11-21')
WINDOW_END    = pd.Timestamp('2022-11-21')

# Consistent palette across every chart in this notebook so the same advertiser
# type reads the same colour everywhere.
GROUP_COLORS = {
    'candidate':           '#1f77b4',   # blue
    'party_org':           '#ff7f0e',   # orange
    'climate':             '#2ca02c',   # green
    'humanitarian_rights': '#d62728',   # red
    'political_advocacy':  '#9467bd',   # purple
    'cost_of_living':      '#8c564b',   # brown
}

## 2. Load v3 and restrict to advocacy

Two filters applied at load:

1. **`match_type IS NULL`** — drop candidate and party_org. We're explicitly looking at the advocacy ecosystem only.
2. **Tag a `group` column** = LDA `category` (`climate`, `humanitarian_rights`, `political_advocacy`, `cost_of_living`).

Cache the result — every subsequent cell scans it.

In [ ]:
v3 = spark.read.parquet(V3_PATH)

# Drop candidate and party_org — focus on advocacy ecosystem only.
v3 = v3.filter(col('match_type').isNull())

# Use category as the group label (climate / humanitarian_rights / political_advocacy / cost_of_living).
v3 = v3.withColumn('group', col('category'))

# Tag pre/post election by ad creation date.
v3 = v3.withColumn(
    'period',
    when(col('ad_creation_date') < lit('2022-05-21'), 'pre')
    .otherwise('post')
)

v3 = v3.cache()
print(f'Advocacy ads in v3: {v3.count():,}')
print('\nGroup distribution:')
v3.groupBy('group').agg(
    spark_count('*').alias('ads'),
    spark_sum('spend_mid').alias('total_spend'),
).orderBy(desc('total_spend')).show(truncate=False)

## 3. Top 20 election spenders (table)

Rank bylines by total spend during the **pre-election** window (Nov 2021 – May 2022) — this is the campaign-period money. For each, also compute post-election spend and a persistence ratio.

Persistence = `post_spend / pre_spend`. Bands:
- `persistence < 0.2` → **election-only** (PAC-style — barely visible after election)
- `0.2 ≤ persistence ≤ 0.8` → **transitional**
- `persistence > 0.8` → **ongoing advocate** (steady-state advertiser)

In [ ]:
# Per-byline pre and post spend totals.
byline_spend = (v3
    .filter(col('spend_mid').isNotNull() & col('bylines').isNotNull())
    .groupBy('bylines')
    .pivot('period', ['pre', 'post'])
    .agg(spark_sum('spend_mid'))
).na.fill(0)

# Modal group per byline — collect counts in Spark, pick max in pandas.
group_per_byline = (v3.filter(col('bylines').isNotNull())
    .groupBy('bylines', 'group').count()
    .toPandas())

primary_group = (group_per_byline
    .sort_values(['bylines', 'count'], ascending=[True, False])
    .drop_duplicates('bylines', keep='first')
    [['bylines', 'group']])

# Bring spend to pandas, merge group, compute persistence, sort.
spend_pdf = byline_spend.toPandas()
spend_pdf.columns = ['bylines', 'pre_spend', 'post_spend']
spend_pdf = spend_pdf.merge(primary_group, on='bylines', how='left')
spend_pdf['total_spend'] = spend_pdf['pre_spend'] + spend_pdf['post_spend']
spend_pdf['persistence'] = spend_pdf['post_spend'] / spend_pdf['pre_spend'].replace(0, pd.NA)
spend_pdf['band'] = pd.cut(
    spend_pdf['persistence'].fillna(0),
    bins=[-0.001, 0.2, 0.8, float('inf')],
    labels=['election_only', 'transitional', 'ongoing']
)

top20 = spend_pdf.sort_values('pre_spend', ascending=False).head(20).reset_index(drop=True)
top20[['bylines', 'group', 'pre_spend', 'post_spend', 'persistence', 'band']]

## 4. Cumulative spend over time (recommended headline chart)

Each top-20 byline's running total of spend through the window. The shape encodes persistence directly:

- **Election-only** advertisers: steep climb pre-election → flat horizontal line after. The curve "elbows" at the election date.
- **Ongoing** advocates: roughly constant slope across the year.

Curves coloured by `group`. End-of-line labels give per-byline identity (lines naturally separate by total spend so labels rarely collide). Election-day vertical line for orientation.

In [ ]:
top20_bylines = top20['bylines'].tolist()

# Weekly aggregation per byline — cached on driver as pandas, reused by every chart below.
weekly = (v3
    .filter(col('bylines').isin(top20_bylines) & col('spend_mid').isNotNull())
    .withColumn('week', date_trunc('week', 'ad_creation_date'))
    .groupBy('week', 'bylines', 'group')
    .agg(spark_sum('spend_mid').alias('spend'))
    .orderBy('week')
    .toPandas())

# Pivot to byline × week (used by stacked/cumulative/heatmap charts below).
pivot = weekly.pivot_table(index='week', columns='bylines', values='spend', aggfunc='sum').fillna(0)
col_order = pivot.sum().sort_values(ascending=False).index.tolist()
pivot = pivot[col_order]
byline_group = dict(zip(top20['bylines'], top20['group']))

# Cumulative spend per byline — the elbow tells the story.
cumulative = pivot.cumsum()

fig, ax = plt.subplots(figsize=(14, 8))
for byline in cumulative.columns:
    grp = byline_group.get(byline, 'unknown')
    color = GROUP_COLORS.get(grp, '#777777')
    ax.plot(cumulative.index, cumulative[byline],
            color=color, alpha=0.85, linewidth=1.6)
    # End-of-line label
    end_val = cumulative[byline].iloc[-1]
    ax.text(cumulative.index[-1] + pd.Timedelta(days=2),
            end_val, ' ' + byline[:35],
            fontsize=7, va='center', color=color, alpha=0.95)

ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.7, linewidth=1.5)
ax.text(ELECTION_DATE + pd.Timedelta(days=2), ax.get_ylim()[1] * 0.95,
        ' Election day', color='red', va='top')
ax.set_title('Cumulative spend by byline — election-only curves "elbow" at the election date')
ax.set_xlabel('Week')
ax.set_ylabel('Cumulative spend ($)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4b. Enrich top 20 with band classifier (persistence + centroid)

Binary pre/post persistence has one failure mode — *off-campaign one-offs* like Shell, whose December 2021 spend gives persistence = 0 even though they're not election-driven. Fix: combine persistence with the centroid (weighted-mean week-from-election) of each advertiser's spending. A persistence-zero advertiser whose centroid is way before the campaign window gets re-classified as off-campaign.

**Four bands**:

| Band | Rule | Interpretation |
|---|---|---|
| `off_campaign` | `persistence < 0.2` **AND** `|centroid_weeks| > 16` | Spent before/after the campaign cycle, not in it (Shell) |
| `election_only` | `persistence < 0.2` (and centroid within 16w) | Spent then stopped at election (Smart Voting, Pharmacy Guild, AEU) |
| `ongoing` | `persistence > 0.8` | Spent at roughly similar rates pre and post (Greenpeace, MSF) |
| `transitional` | else | Tapered after election (Climate 200, Amnesty, ACF) |

**Two supplementary metrics carried alongside**:

- **Election Concentration (EC) index**: triangular-weighted concentration of spend around election day. Range 0–1 with 0.5 as the uniform baseline. Used by the EC scatter chart (section 9b) but no longer drives band classification.
- **Centroid (weeks-from-election)**: weighted-mean week of spending. Negative = pre-election; positive = post-election; near 0 = election week.

In [ ]:
import numpy as np

HALF_WIDTH_WEEKS = 26   # triangle window: 0 at edges, 1 at election

# Reuse the `weekly` DataFrame from cell 9. Add triangle weights for EC index.
weekly['weeks_from_election'] = (weekly['week'] - ELECTION_DATE).dt.days / 7
weekly['weight'] = np.clip(
    1.0 - np.abs(weekly['weeks_from_election']) / HALF_WIDTH_WEEKS,
    a_min=0.0, a_max=None,
)
weekly['weighted_spend'] = weekly['weight'] * weekly['spend']

# Per-byline EC index and weighted-mean week-from-election (centroid).
ec_calc = (weekly.groupby('bylines')
    .apply(lambda x: pd.Series({
        'ec_index':       (x['weighted_spend'].sum() / x['spend'].sum()) if x['spend'].sum() > 0 else 0.0,
        'centroid_weeks': (x['weeks_from_election'] * x['spend']).sum() / x['spend'].sum() if x['spend'].sum() > 0 else 0.0,
    }))
    .reset_index())

top20 = top20.merge(ec_calc, on='bylines', how='left')

# Drop the old persistence-only `band` column if it's still there from cell 7.
if 'band' in top20.columns:
    top20 = top20.drop(columns=['band'])
if 'ec_band' in top20.columns:
    top20 = top20.drop(columns=['ec_band'])

# Combined band classifier — persistence as primary signal, centroid as off-campaign override.
CENTROID_WINDOW = 16   # weeks. Beyond this from election → off-campaign

def classify(row):
    p = row['persistence'] if pd.notna(row['persistence']) else 0.0
    c = abs(row['centroid_weeks']) if pd.notna(row['centroid_weeks']) else 0.0
    if p < 0.2 and c > CENTROID_WINDOW:
        return 'off_campaign'
    if p < 0.2:
        return 'election_only'
    if p > 0.8:
        return 'ongoing'
    return 'transitional'

top20['band'] = top20.apply(classify, axis=1)
top20['band'] = pd.Categorical(
    top20['band'],
    categories=['election_only', 'transitional', 'ongoing', 'off_campaign'],
    ordered=True,
)

# Display table — primary sort by band, secondary by total spend descending.
display_cols = ['bylines', 'group', 'pre_spend', 'post_spend',
                'persistence', 'ec_index', 'centroid_weeks', 'band']
top20.sort_values(['band', 'total_spend'], ascending=[True, False])[display_cols]

## 5. Stacked area — absolute weekly spend

Same data, stacked by byline. Each byline gets a distinct colour (tab20 palette) since colour-by-group caused repeated colours and was unreadable. Top edge of the stack = total weekly spend across the top 20.

Best for showing *when total advocacy spending happened* and the relative composition. Still bursty week-to-week, but the aggregate envelope shows the election-period concentration clearly.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
pivot.plot.area(ax=ax, alpha=0.85, linewidth=0, colormap='tab20')
ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.9, linewidth=1.5)
ax.text(ELECTION_DATE + pd.Timedelta(days=2), ax.get_ylim()[1] * 0.95,
        ' Election day', color='red', va='top')
ax.set_title('Weekly spend — stacked area, top 20 advocacy advertisers')
ax.set_ylabel('Spend ($)')
ax.set_xlabel('Week')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Small multiples — grouped by band

Each top-20 advertiser gets its own panel, **grouped into four figures** by band (`election_only` / `transitional` / `ongoing` / `off_campaign`). Shared x-axis across all panels in a band (full Nov 2021 → Nov 2022 window) so timing is directly comparable. Each panel has its own y-axis (no `sharey`) so small-budget advertisers are visible.

Within each figure, panels are sorted by total spend descending. Panel title shows `(group, total spend, EC)` for context. Coloured by primary `group`.

In [ ]:
import math

# Full time window for sharex.
x_min = weekly['week'].min()
x_max = weekly['week'].max()

for band_name in ['election_only', 'transitional', 'ongoing', 'off_campaign']:
    band_data = top20[top20['band'] == band_name].sort_values('total_spend', ascending=False).reset_index(drop=True)
    n = len(band_data)
    if n == 0:
        print(f'No advertisers in band "{band_name}".\n')
        continue

    ncols = min(4, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.5 * nrows),
                             squeeze=False, sharex=True, sharey=False)
    axes = axes.flatten()

    for i, row in band_data.iterrows():
        ax = axes[i]
        byline = row['bylines']
        grp = row['group']
        color = GROUP_COLORS.get(grp, '#777777')
        sub = weekly[weekly['bylines'] == byline].sort_values('week')
        ax.plot(sub['week'], sub['spend'], color=color, linewidth=1.3)
        ax.fill_between(sub['week'], sub['spend'], color=color, alpha=0.3)
        ax.axvline(ELECTION_DATE, linestyle='--', color='red', alpha=0.5)
        ax.set_title(f'{byline[:32]}\n({grp}, ${row["total_spend"]:,.0f}, EC={row["ec_index"]:.2f})', fontsize=8)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', labelsize=7)
        ax.grid(alpha=0.3)
        ax.set_xlim(x_min, x_max)

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Weekly spend — {band_name} band ({n} advertisers)', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

## 7. Heatmap — bylines × weeks

20 rows (bylines, sorted by band then persistence ascending so election_only / off_campaign appear at top, ongoing at bottom), 52 columns (weeks). Colour = log-scaled weekly spend.

Election-driven patterns show a dense bright stripe at the election week. Off-campaign patterns show a bright stripe well away from election day. Ongoing patterns show colour evenly distributed across the year.

Dense overview in a single chart.

In [ ]:
import numpy as np

# Sort by band order (election_only first), then by persistence ascending within band
# so the most extreme cliff-edge advertisers sit at the top of each band.
byline_order = top20.sort_values(['band', 'persistence'], ascending=[True, True]) \
    .reset_index(drop=True)['bylines'].tolist()
heat = pivot[byline_order].T

# Log-scaled colour with floor to avoid log(0).
heat_log = np.log10(heat + 1)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(heat_log, aspect='auto', cmap='viridis', interpolation='nearest')

ax.set_yticks(range(len(byline_order)))
ax.set_yticklabels([b[:40] for b in byline_order], fontsize=8)

week_labels = [w.strftime('%Y-%m') for w in heat.columns]
ax.set_xticks(range(0, len(heat.columns), 4))
ax.set_xticklabels(week_labels[::4], rotation=45, ha='right', fontsize=7)

# Election-day marker.
election_indices = [i for i, w in enumerate(heat.columns) if w >= ELECTION_DATE]
if election_indices:
    ax.axvline(election_indices[0], color='red', linestyle='--', alpha=0.85, linewidth=1.5)

fig.colorbar(im, ax=ax, label='log10(spend + 1)')
ax.set_title('Weekly spend heatmap — rows grouped by band (election_only top, ongoing bottom)')
ax.set_xlabel('Week')
plt.tight_layout()
plt.show()

## 8. Dumbbell — pre vs post per byline

Same data as the pre/post bar chart further down, different visual. Each byline is a horizontal line from pre-election spend (blue) to post-election spend (red). Long lines = high persistence gap (election-only); short lines = ongoing.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
y_pos = list(range(len(top20)))

# Connecting line (grey)
for i, row in top20.iterrows():
    ax.plot([row['pre_spend'], row['post_spend']], [i, i],
            color='#cccccc', linewidth=1.5, zorder=1)

# Endpoint markers
ax.scatter(top20['pre_spend'], y_pos, color='#1f77b4', s=90, zorder=3,
           label='pre-election', edgecolors='black', linewidth=0.4)
ax.scatter(top20['post_spend'], y_pos, color='#d62728', s=90, zorder=3,
           label='post-election', edgecolors='black', linewidth=0.4)

ax.set_yticks(y_pos)
ax.set_yticklabels([b[:50] for b in top20['bylines']], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Spend ($)')
ax.set_title('Dumbbell: pre vs post-election spend per byline (long line = election-only)')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Persistence scatter — election-only vs ongoing

Each top-20 byline as a point. X-axis = total spend (log scale). Y-axis = persistence ratio. Coloured by `group`. Quadrants:

- Top-left: high spend, low persistence → **election-only PAC-style**
- Top-right: high spend, high persistence → **ongoing advocate**
- Bottom-left: low spend, low persistence → one-off campaign
- Bottom-right: low spend, high persistence → small persistent advertiser

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for grp, sub in top20.groupby('group'):
    ax.scatter(
        sub['total_spend'], sub['persistence'].fillna(0),
        s=120, alpha=0.7,
        color=GROUP_COLORS.get(grp, '#777777'),
        label=grp, edgecolors='black', linewidth=0.5,
    )
for _, row in top20.iterrows():
    ax.annotate(
        row['bylines'][:25],
        (row['total_spend'], row['persistence'] if pd.notna(row['persistence']) else 0),
        fontsize=7, alpha=0.85, xytext=(5, 5), textcoords='offset points',
    )

# Reference lines for the persistence bands
ax.axhline(0.2, linestyle=':', color='gray', alpha=0.6)
ax.axhline(0.8, linestyle=':', color='gray', alpha=0.6)
ax.text(ax.get_xlim()[1] * 0.95, 0.1, 'election-only',
        color='gray', ha='right', fontsize=8)
ax.text(ax.get_xlim()[1] * 0.95, 0.5, 'transitional',
        color='gray', ha='right', fontsize=8)
ax.text(ax.get_xlim()[1] * 0.95, 1.0, 'ongoing',
        color='gray', ha='right', fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Total spend, log scale ($)')
ax.set_ylabel('Persistence ratio (post / pre)')
ax.set_title('Top 20 non-government election spenders — persistence vs spend')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9b. Election Concentration scatter — election-driven vs distributed

Same shape as the persistence scatter, but Y-axis is now the **Election Concentration (EC) index** — the triangular-weighted share of spending concentrated near election day (window half-width = 26 weeks).

The uniform baseline is 0.5: a steady-state advertiser distributing their spending evenly across the year scores exactly 0.5. So the interpretation is anchored:

- **EC > 0.65**: more concentrated than uniform → election-driven (top-right with high spend = PAC-style)
- **EC ≈ 0.5**: similar to uniform → distributed advocacy (Greenpeace-style steady-state)
- **EC < 0.4**: less concentrated than uniform → off-campaign one-off (Shell's December spike)

This separates *concentration* from *timing direction* — Shell's pre-election spike gets correctly identified as low-EC because it's far from election day, while UAP's pre-election spike scores well above the uniform baseline.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for grp, sub in top20.groupby('group'):
    ax.scatter(
        sub['total_spend'], sub['ec_index'],
        s=130, alpha=0.7,
        color=GROUP_COLORS.get(grp, '#777777'),
        label=grp, edgecolors='black', linewidth=0.5,
    )

for _, row in top20.iterrows():
    ax.annotate(
        row['bylines'][:25],
        (row['total_spend'], row['ec_index']),
        fontsize=7, alpha=0.85,
        xytext=(5, 5), textcoords='offset points',
    )

# Reference: uniform-distribution baseline at 0.5
ax.axhline(0.5, linestyle=':', color='gray', alpha=0.7)
ax.text(ax.get_xlim()[1] * 0.95, 0.51, 'uniform baseline',
        color='gray', ha='right', fontsize=8, va='bottom')

ax.set_xscale('log')
ax.set_xlabel('Total spend, log scale ($)')
ax.set_ylabel(f'Election Concentration Index (triangle, half-width = {HALF_WIDTH_WEEKS}w)')
ax.set_title('Election concentration vs total spend — high EC = election-driven, low EC = off-campaign')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Pre vs post spend — bar chart

Direct side-by-side comparison for each top-20 byline. Election-only advertisers have near-zero post bars; ongoing advocates have comparable pre/post bars.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
y = range(len(top20))
bar_height = 0.4
ax.barh([i - bar_height/2 for i in y], top20['pre_spend'],
        height=bar_height, label='pre-election', color='#1f77b4', alpha=0.85)
ax.barh([i + bar_height/2 for i in y], top20['post_spend'],
        height=bar_height, label='post-election', color='#d62728', alpha=0.85)

ax.set_yticks(list(y))
ax.set_yticklabels([b[:50] for b in top20['bylines']], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Spend ($)')
ax.set_title('Pre-election vs post-election spend — top 20')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Band summary table

Roll the top-20 up to bands — count, total spend, and the persistence/EC means per band. Quantifies the relative size of each spending pattern.

In [ ]:
band_summary = top20.groupby('band', observed=True).agg(
    n_advertisers=('bylines', 'count'),
    pre_spend=('pre_spend', 'sum'),
    post_spend=('post_spend', 'sum'),
    total_spend=('total_spend', 'sum'),
    mean_persistence=('persistence', 'mean'),
    mean_ec=('ec_index', 'mean'),
).reset_index()

band_summary['total_share'] = band_summary['total_spend'] / band_summary['total_spend'].sum()

band_summary

## 12. Discussion

*(Filled in after running the analysis — sketching headline findings)*

**Three findings expected**:

1. **Among non-party, non-government advocacy advertisers, four temporal patterns are observable**, classified by persistence and centroid:
   - **`election_only`** — clearly election-aligned campaigns (Smart Voting, Solutions for Australia, Pharmacy Guild's Affordable Medicines Now, AEU's election push, GetUp!, It's Not A Race) that ramped up pre-election and stopped at the polling date.
   - **`transitional`** — campaigns that bumped around the election but tapered rather than cliff-edged (Climate 200, Amnesty, ACF, Thrive By Five, UNHCR).
   - **`ongoing`** — year-round advocates whose post-election activity equals or exceeds pre-election activity (Greenpeace, MSF, Plan International, Australian Unions).
   - **`off_campaign`** — large but isolated spending events that aren't aligned with the election cycle (Shell's December 2021 spike).

2. **Election-only is the most populous and the second-largest by spend.** Around 7–10 of the top 20 advertisers cliff-edged at the election. They include the closest Australian equivalents to US PAC-style temporal-spending patterns — Smart Voting, Solutions for Australia, and Advance Australia among them.

3. **Greenpeace is the singular dominant ongoing advocate** — total spend in the top tier, persistence near 1, centroid almost exactly on election day. Their year-round campaigning is the temporal baseline against which everyone else's election-alignment is measurable.

**Main message**: the non-party, non-government advocacy ecosystem is not monolithic. It splits cleanly into four behavioural patterns, with the largest cluster (election-only) reproducing US-style PAC dynamics at smaller scale. Combining persistence (post/pre spend ratio) with the centroid of spending activity separates these patterns reliably — neither metric alone suffices, but together they correctly classify off-campaign one-offs like Shell as well as the more obvious election-driven and steady-state cases.